# MCP Tech News Client

This notebook connects to the `tech_news_server.py` MCP server.

## Before running this notebook:

Start the server in a terminal:
```bash
python tech_news_server.py
```

Or use this notebook to spawn the server as a subprocess.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [ ]:
# Configure connection to the tech_news server
server_params = StdioServerParameters(
    command="python",
    args=["tech_news_server.py"],
    env=None
)

In [ ]:
class MCPClient:
    """A simple MCP client wrapper for notebook usage."""
    
    def __init__(self, server_params: StdioServerParameters):
        self.server_params = server_params
        self.session = None
        self._read = None
        self._write = None
        self._client_cm = None
        self._session_cm = None
    
    async def connect(self):
        """Connect to the MCP server."""
        self._client_cm = stdio_client(self.server_params)
        self._read, self._write = await self._client_cm.__aenter__()
        
        self._session_cm = ClientSession(self._read, self._write)
        self.session = await self._session_cm.__aenter__()
        
        await self.session.initialize()
        print("Connected to MCP server")
        return self
    
    async def disconnect(self):
        """Disconnect from the MCP server."""
        # Exit in reverse order of entry
        if self._session_cm:
            try:
                await self._session_cm.__aexit__(None, None, None)
            except Exception as e:
                print(f"Warning closing session: {e}")
            self._session_cm = None
            self.session = None
        
        if self._client_cm:
            try:
                await self._client_cm.__aexit__(None, None, None)
            except Exception as e:
                print(f"Warning closing client: {e}")
            self._client_cm = None
        
        print("Disconnected from MCP server")
    
    async def list_tools(self):
        """List available tools from the server."""
        tools = await self.session.list_tools()
        return tools.tools
    
    async def call_tool(self, name: str, arguments: dict = None):
        """Call a tool on the server."""
        result = await self.session.call_tool(name, arguments or {})
        return result

In [ ]:
# Create and connect the client
client = MCPClient(server_params)
await client.connect()

In [ ]:
# List available tools
tools = await client.list_tools()
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

In [ ]:
# Get available news sources
result = await client.call_tool("list_sources")
print("Available sources:", result)

In [ ]:
# Fetch tech news from arstechnica
result = await client.call_tool("get_tech_news", {"source": "arstechnica"})
print("News:", result)

# Ollama Agent with MCP Tools

Now let's create an agent that uses Ollama to decide when to call MCP tools.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# Create LangChain tools that wrap the MCP client calls
@tool
async def get_tech_news(source: str) -> str:
    """Fetches the latest news from a specific tech news source.
    
    Args:
        source: Name of the news source (e.g., 'arstechnica')
    """
    result = await client.call_tool("get_tech_news", {"source": source})
    return result.content[0].text if result.content else "No content"

@tool
async def list_sources() -> str:
    """Lists all available news sources."""
    result = await client.call_tool("list_sources")
    return result.content[0].text if result.content else "No sources"

# List of tools for the agent
langchain_tools = [get_tech_news, list_sources]

# Create Ollama model with tools bound
llm = ChatOllama(model="llama3.2")
llm_with_tools = llm.bind_tools(langchain_tools)

print("LangChain tools bound to Ollama:")
for t in langchain_tools:
    print(f"  - {t.name}: {t.description}")

In [ ]:
async def run_agent(user_message: str, max_iterations: int = 10):
    """
    Run an agent loop with LangChain Ollama that can use MCP tools.
    
    Args:
        user_message: The user's question or request
        max_iterations: Maximum number of tool-calling iterations
    """
    # Map tool names to tool functions
    tools_by_name = {t.name: t for t in langchain_tools}
    
    messages = [HumanMessage(content=user_message)]
    print(f"User: {user_message}\n")
    
    for _ in range(max_iterations):
        # Call LLM with tools
        response = await llm_with_tools.ainvoke(messages)
        messages.append(response)
        
        # Check if there are tool calls
        if not response.tool_calls:
            # No tool calls, return final response
            print(f"Assistant: {response.content}")
            return response.content
        
        # Process each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            print(f"[Calling tool: {tool_name}({tool_args})]")
            
            # Execute the tool
            tool_fn = tools_by_name[tool_name]
            result = await tool_fn.ainvoke(tool_args)
            
            print(f"[Tool result: {result[:200]}...]\n" if len(str(result)) > 200 else f"[Tool result: {result}]\n")
            
            # Add tool result to messages
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    
    return "Max iterations reached"

In [ ]:
# Run the agent with a question
await run_agent("What are the latest tech news from arstechnica?")

In [ ]:
# Another example - asking about available sources
await run_agent("What news sources do you have access to?")

In [ ]:
# Disconnect when done
await client.disconnect()